# Test based on https://github.com/NathanWycoff/prox_alasso/blob/main/python/ada_sim.py

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from adaptivelasso.skcompat import MAPAdaptiveLasso
from adaptivelasso.utils import generate_data, report_model_results
import numpy as np

## Compare MAPAdaptiveLasso with LassoCV

In [2]:
# Example for normal likelihood
N, P = 1000, 20
beta = np.zeros(P)
beta[:5] = 2  # 5 informative features
sigma = 1.0
lik = 'normal'

X, y = generate_data(lik, N, P, beta, sigma=sigma, random_state=42)

# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=5000, track=False, tol=1e-4, cv=5))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

# Use a pipeline for scaling and LassoCV
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=0, max_iter=5000, tol=1e-4))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

ES stop!
ES stop!
Estimated coefficients: [ 1.34543251e+00  1.34544432e+00  1.22827788e+00  1.36146619e+00
  1.36443837e+00  3.38394358e-02  1.65747310e-01  1.51566283e-01
  6.55682827e-02 -2.99462744e-02 -9.66660774e-02  1.14003002e-01
  7.85721607e-02  1.30524943e-01 -8.36840997e-02 -5.43380968e-02
  0.00000000e+00 -0.00000000e+00  1.37756660e-01 -3.16348111e-04]
R²: 0.5737
RMSE: 0.3241
Estimated coefficients: [ 0.16488544  0.1550696   0.14862804  0.16592588  0.16575047 -0.
  0.01683911  0.01516307  0.0040984  -0.0040288  -0.          0.01580763
 -0.          0.00102194  0.         -0.00028995  0.00109467  0.00600582
  0.01435396  0.        ]
R²: 0.4902
RMSE: 0.3544


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from adaptivelasso.skcompat import MAPAdaptiveLasso

# Assume generate_data and report_model_results are already defined as above

likelihoods = ['normal', 'cauchy', 'bernoulli', 'poisson', 'zinb', 'nb']
results = []
max_iters = 3000  # Set a maximum number of iterations
tol = 1e-4  # Set a tolerance for convergence
cv = 5

for lik in likelihoods:
    print(f"Testing likelihood: {lik}")
    N, P = 1000, 20
    true_coefs = np.zeros(P)
    true_coefs[:3] = 1.0
    sigma = 1.0

    X, y = generate_data(lik, N, P, beta=true_coefs, sigma=sigma, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

    # MAPAdaptiveLasso
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=max_iters, track=True, tol=tol, cv=cv))
    ])
    pipe.fit(X_train, y_train)
    y_pred_adapt = pipe.predict(X_test)
    if lik == 'cauchy':
        residuals_adapt = y_test - y_pred_adapt
        mad_adapt = np.median(np.abs(residuals_adapt - np.median(residuals_adapt)))
        r2_adapt = r2_score(y_test, y_pred_adapt)
        rmse_adapt = np.nan
    else:
        r2_adapt = r2_score(y_test, y_pred_adapt)
        rmse_adapt = np.sqrt(mean_squared_error(y_test, y_pred_adapt))
        mad_adapt = np.nan

    # LassoCV
    pipe_lasso = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', LassoCV(cv=5, random_state=0, max_iter=max_iters))
    ])
    pipe_lasso.fit(X_train, y_train)
    y_pred_lasso = pipe_lasso.predict(X_test)
    if lik == 'bernoulli':
        y_pred_lasso_bin = (y_pred_lasso > 0.5).astype(int)
        r2_lasso = r2_score(y_test, y_pred_lasso_bin)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso_bin))
        mad_lasso = np.nan
    elif lik == 'cauchy':
        residuals_lasso = y_test - y_pred_lasso
        mad_lasso = np.median(np.abs(residuals_lasso - np.median(residuals_lasso)))
        r2_lasso = r2_score(y_test, y_pred_lasso)
        rmse_lasso = np.nan
    else:
        r2_lasso = r2_score(y_test, y_pred_lasso)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
        mad_lasso = np.nan

    # Decide which is better for each metric
    if lik == 'cauchy':
        better_mad = (
            'AdaptiveLasso' if mad_adapt < mad_lasso else
            'LassoCV' if mad_lasso < mad_adapt else
            'Tie'
        )
        better_rmse = np.nan
    else:
        better_mad = np.nan
        better_rmse = (
            'AdaptiveLasso' if rmse_adapt < rmse_lasso else
            'LassoCV' if rmse_lasso < rmse_adapt else
            'Tie'
        )

    if np.isnan(r2_lasso):
        better_r2 = 'AdaptiveLasso only'
    else:
        better_r2 = (
            'AdaptiveLasso' if r2_adapt > r2_lasso else
            'LassoCV' if r2_lasso > r2_adapt else
            'Tie'
        )

    results.append({
        'likelihood': lik,
        'R2_AdaptiveLasso': r2_adapt,
        'RMSE_AdaptiveLasso': rmse_adapt,
        'MAD_AdaptiveLasso': mad_adapt,
        'R2_LassoCV': r2_lasso,
        'RMSE_LassoCV': rmse_lasso,
        'MAD_LassoCV': mad_lasso,
        'Better_R2': better_r2,
        'Better_RMSE': better_rmse,
        'Better_MAD': better_mad,
    })

df_results = pd.DataFrame(results)
print(df_results)

Testing likelihood: normal
ES stop!
ES stop!
ES stop!
ES stop!
ES stop!
Testing likelihood: cauchy
ES stop!
Testing likelihood: bernoulli
ES stop!
ES stop!
Testing likelihood: poisson
ES stop!
ES stop!
ES stop!
Testing likelihood: zinb
ES stop!
ES stop!
Testing likelihood: nb
ES stop!
  likelihood  R2_AdaptiveLasso  RMSE_AdaptiveLasso  MAD_AdaptiveLasso  \
0     normal          0.722259            0.961862                NaN   
1     cauchy          0.020460                 NaN           0.928932   
2  bernoulli          0.192079            0.448612                NaN   
3    poisson          0.973212            2.188954                NaN   
4       zinb          0.352073           11.999097                NaN   
5         nb          0.506795           13.704425                NaN   

   R2_LassoCV  RMSE_LassoCV  MAD_LassoCV      Better_R2    Better_RMSE  \
0    0.736476      0.936920          NaN        LassoCV        LassoCV   
1    0.000429           NaN     1.374673  AdaptiveLass